<a href="https://colab.research.google.com/github/josemanuelmontanoortiz978-droid/PROYECTOS-ED1/blob/main/Ejercicios_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from __future__ import annotations
from abc import ABC, abstractmethod
from typing import Any, Optional
import ctypes


class Nodo:
    """Nodo individual para estructuras enlazadas."""

    def __init__(self, dato: Any, siguiente: Optional[Nodo] = None) -> None:
        self.dato: Any = dato
        self.siguiente: Optional[Nodo] = siguiente


class ADTPila(ABC):
    """Interfaz abstracta (Tipo Abstracto de Datos) para una Pila LIFO."""

    @abstractmethod
    def apilar(self, elemento: Any) -> None:
        """Inserta un elemento en la cima de la pila."""
        pass

    @abstractmethod
    def desapilar(self) -> Any:
        """Extrae y retorna el elemento en la cima de la pila."""
        pass

    @abstractmethod
    def cima(self) -> Any:
        """Retorna el elemento en la cima sin extraerlo."""
        pass

    @abstractmethod
    def esta_vacia(self) -> bool:
        """Indica si la pila no contiene elementos."""
        pass

    @abstractmethod
    def __len__(self) -> int:
        """Retorna la cantidad de elementos almacenados."""
        pass


class PilaArray(ADTPila):
    """Implementación estática de ADTPila usando un bloque contiguo ctypes."""

    def __init__(self, capacidad: int = 100) -> None:
        if capacidad <= 0:
            raise ValueError("La capacidad debe ser mayor a 0.")
        self._capacidad: int = capacidad
        self._datos: ctypes.Array[Any] = (ctypes.py_object * capacidad)()
        self._tope: int = -1

    def apilar(self, elemento: Any) -> None:
        if self._tope + 1 >= self._capacidad:
            raise OverflowError("Desbordamiento de pila (Stack Overflow): capacidad llena.")
        self._tope += 1
        self._datos[self._tope] = elemento

    def desapilar(self) -> Any:
        if self.esta_vacia():
            raise IndexError("Subdesbordamiento de pila (Stack Underflow): pila vacía.")
        valor: Any = self._datos[self._tope]
        self._datos[self._tope] = None  # Liberar referencia para el Garbage Collector
        self._tope -= 1
        return valor

    def cima(self) -> Any:
        if self.esta_vacia():
            raise IndexError("La pila está vacía.")
        return self._datos[self._tope]

    def esta_vacia(self) -> bool:
        return self._tope == -1

    def __len__(self) -> int:
        return self._tope + 1


class PilaLista(ADTPila):
    """Implementación dinámica de ADTPila usando nodos enlazados."""

    def __init__(self) -> None:
        self._cima: Optional[Nodo] = None
        self._tamanio: int = 0

    def apilar(self, elemento: Any) -> None:
        self._cima = Nodo(elemento, self._cima)
        self._tamanio += 1

    def desapilar(self) -> Any:
        if self.esta_vacia():
            raise IndexError("Subdesbordamiento de pila (Stack Underflow): pila vacía.")
        assert self._cima is not None
        valor: Any = self._cima.dato
        self._cima = self._cima.siguiente
        self._tamanio -= 1
        return valor

    def cima(self) -> Any:
        if self.esta_vacia():
            raise IndexError("La pila está vacía.")
        assert self._cima is not None
        return self._cima.dato

    def esta_vacia(self) -> bool:
        return self._cima is None

    def __len__(self) -> int:
        return self._tamanio


def ejecutar_flujo_pila(pila: ADTPila, operaciones: list[str]) -> list[str]:
    """Código cliente desacoplado: funciona idéntico con cualquier ADTPila."""
    historial_deshacer: list[str] = []
    for accion in operaciones:
        pila.apilar(accion)

    print(f"Implementación activa: {type(pila).__name__} | Cima actual: {pila.cima()}")

    while not pila.esta_vacia():
        historial_deshacer.append(pila.desapilar())

    return historial_deshacer


# --- INTERCAMBIO TRANSPARENTE DE IMPLEMENTACIÓN ---
acciones: list[str] = ["Escribir título", "Insertar tabla", "Negrita en encabezado"]

pila_estatica: ADTPila = PilaArray(capacidad=10)
pila_dinamica: ADTPila = PilaLista()

print("Resultado PilaArray:", ejecutar_flujo_pila(pila_estatica, acciones))
print("Resultado PilaLista:", ejecutar_flujo_pila(pila_dinamica, acciones))

Implementación activa: PilaArray | Cima actual: Negrita en encabezado
Resultado PilaArray: ['Negrita en encabezado', 'Insertar tabla', 'Escribir título']
Implementación activa: PilaLista | Cima actual: Negrita en encabezado
Resultado PilaLista: ['Negrita en encabezado', 'Insertar tabla', 'Escribir título']


ESTATICO VS DINAMICO

In [2]:
class ArrayEstatico:
    """Arreglo de tamaño fijo con control explícito de desbordamiento e índices."""

    def __init__(self, capacidad: int) -> None:
        if capacidad <= 0:
            raise ValueError("La capacidad debe ser positiva.")
        self._capacidad: int = capacidad
        self._datos: ctypes.Array[Any] = (ctypes.py_object * capacidad)()
        self._tamanio: int = 0

    def insertar_final(self, valor: Any) -> None:
        """Inserta al final en O(1) si hay espacio disponible."""
        if self._tamanio >= self._capacidad:
            raise OverflowError(f"ArrayEstatico lleno (capacidad={self._capacidad}).")
        self._datos[self._tamanio] = valor
        self._tamanio += 1

    def insertar_en(self, indice: int, valor: Any) -> None:
        """Inserta en posición arbitraria desplazando elementos a la derecha: O(n)."""
        if self._tamanio >= self._capacidad:
            raise OverflowError(f"ArrayEstatico lleno (capacidad={self._capacidad}).")
        if not (0 <= indice <= self._tamanio):
            raise IndexError("Índice fuera de rango.")
        for i in range(self._tamanio, indice, -1):
            self._datos[i] = self._datos[i - 1]
        self._datos[indice] = valor
        self._tamanio += 1

    def obtener(self, indice: int) -> Any:
        """Acceso directo por desplazamiento de memoria en O(1)."""
        if not (0 <= indice < self._tamanio):
            raise IndexError("Índice fuera de rango.")
        return self._datos[indice]

    def eliminar_en(self, indice: int) -> Any:
        """Elimina elemento y compacta hacia la izquierda en O(n)."""
        if not (0 <= indice < self._tamanio):
            raise IndexError("Índice fuera de rango.")
        eliminado: Any = self._datos[indice]
        for i in range(indice, self._tamanio - 1):
            self._datos[i] = self._datos[i + 1]
        self._datos[self._tamanio - 1] = None
        self._tamanio -= 1
        return eliminado

    def __len__(self) -> int:
        return self._tamanio


class ListaDinamica:
    """Lista simplemente enlazada con punteros cabeza y cola."""

    def __init__(self) -> None:
        self._cabeza: Optional[Nodo] = None
        self._cola: Optional[Nodo] = None
        self._tamanio: int = 0

    def insertar_inicio(self, valor: Any) -> None:
        """Inserta al inicio redirigiendo cabeza en O(1)."""
        nuevo: Nodo = Nodo(valor, self._cabeza)
        self._cabeza = nuevo
        if self._cola is None:
            self._cola = nuevo
        self._tamanio += 1

    def insertar_final(self, valor: Any) -> None:
        """Inserta al final usando el puntero _cola en O(1)."""
        nuevo: Nodo = Nodo(valor)
        if self._cabeza is None:
            self._cabeza = nuevo
            self._cola = nuevo
        else:
            assert self._cola is not None
            self._cola.siguiente = nuevo
            self._cola = nuevo
        self._tamanio += 1

    def obtener(self, indice: int) -> Any:
        """Acceso secuencial recorriendo nodos desde cabeza: O(n)."""
        if not (0 <= indice < self._tamanio):
            raise IndexError("Índice fuera de rango.")
        actual: Optional[Nodo] = self._cabeza
        for _ in range(indice):
            assert actual is not None
            actual = actual.siguiente
        assert actual is not None
        return actual.dato

    def eliminar_en(self, indice: int) -> Any:
        """Elimina el nodo en la posición dada tras recorrer enlaces: O(n)."""
        if not (0 <= indice < self._tamanio):
            raise IndexError("Índice fuera de rango.")
        assert self._cabeza is not None

        if indice == 0:
            eliminado: Any = self._cabeza.dato
            self._cabeza = self._cabeza.siguiente
            if self._cabeza is None:
                self._cola = None
            self._tamanio -= 1
            return eliminado

        anterior: Optional[Nodo] = self._cabeza
        for _ in range(indice - 1):
            assert anterior is not None
            anterior = anterior.siguiente

        assert anterior is not None and anterior.siguiente is not None
        eliminado = anterior.siguiente.dato
        if anterior.siguiente == self._cola:
            self._cola = anterior
        anterior.siguiente = anterior.siguiente.siguiente
        self._tamanio -= 1
        return eliminado

    def __len__(self) -> int:
        return self._tamanio

SIMULACION DE 50 ALUMNOS

In [ ]:
import random
import sys
import time
from dataclasses import dataclass, asdict


@dataclass
class Estudiante:
    """Representa el registro académico de un estudiante."""
    nombre: str
    nota: float
    grupo: str


def generar_estudiantes(cantidad: int = 50) -> list[Estudiante]:
    """Genera registros simulados deterministas/aleatorios de estudiantes."""
    nombres_base: list[str] = ["Ana", "Luis", "Carla", "Mateo", "Sofía", "Diego", "Valeria", "Jorge"]
    apellidos_base: list[str] = ["Gómez", "Pérez", "Rojas", "Vargas", "Silva", "Mendoza", "Castro", "Ortiz"]
    grupos: list[str] = ["G1", "G2", "G3"]

    registros: list[Estudiante] = []
    for i in range(1, cantidad + 1):
        nombre_completo: str = f"{random.choice(nombres_base)} {random.choice(apellidos_base)} #{i}"
        nota: float = round(random.uniform(51.0, 100.0), 1)
        grupo: str = random.choice(grupos)
        registros.append(Estudiante(nombre=nombre_completo, nota=nota, grupo=grupo))
    return registros


def calcular_memoria_lista_dinamica(lista: ListaDinamica) -> int:
    """Suma el tamaño base de la estructura más cada objeto Nodo y Estudiante."""
    total_bytes: int = sys.getsizeof(lista)
    actual: Optional[Nodo] = lista._cabeza
    while actual is not None:
        total_bytes += sys.getsizeof(actual) + sys.getsizeof(actual.dato)
        actual = actual.siguiente
    return total_bytes


def calcular_memoria_array_estatico(arreglo: ArrayEstatico) -> int:
    """Suma el tamaño del contenedor, el buffer ctypes y los objetos referenciados."""
    total_bytes: int = sys.getsizeof(arreglo) + sys.getsizeof(arreglo._datos)
    for i in range(len(arreglo)):
        total_bytes += sys.getsizeof(arreglo.obtener(i))
    return total_bytes


# --- EJECUCIÓN DEL BENCHMARK ---
random.seed(42)
lote_estudiantes: list[Estudiante] = generar_estudiantes(50)

arr_estatico: ArrayEstatico = ArrayEstatico(capacidad=50)
lista_dinamica: ListaDinamica = ListaDinamica()

# 1. Tiempo de inserción
t0: float = time.perf_counter()
for est in lote_estudiantes:
    arr_estatico.insertar_final(est)
t_array: float = (time.perf_counter() - t0) * 1_000_000  # microsegundos

t0 = time.perf_counter()
for est in lote_estudiantes:
    lista_dinamica.insertar_final(est)
t_lista: float = (time.perf_counter() - t0) * 1_000_000  # microsegundos

# 2. Facilidad de búsqueda (acceso al estudiante #35 vs búsqueda por nombre)
t0 = time.perf_counter()
est_idx_array: Estudiante = arr_estatico.obtener(34)
t_busqueda_idx_array: float = (time.perf_counter() - t0) * 1_000_000

t0 = time.perf_counter()
est_idx_lista: Estudiante = lista_dinamica.obtener(34)
t_busqueda_idx_lista: float = (time.perf_counter() - t0) * 1_000_000

# 3. Memoria aproximada
mem_array: int = calcular_memoria_array_estatico(arr_estatico)
mem_lista: int = calcular_memoria_lista_dinamica(lista_dinamica)

print(f"{'Métrica':<32} | {'ArrayEstatico':<18} | {'ListaDinamica':<18}")
print("-" * 74)
print(f"{'Tiempo inserción (50 reg.)':<32} | {t_array:>14.2f} µs | {t_lista:>14.2f} µs")
print(f"{'Acceso por índice [34]':<32} | {t_busqueda_idx_array:>14.2f} µs | {t_busqueda_idx_lista:>14.2f} µs")
print(f"{'Memoria aproximada (bytes)':<32} | {mem_array:>15} B | {mem_lista:>15} B")

JSON VS PICKLE

In [ ]:
import json
import pickle


def guardar_json(ruta: str, estudiantes: ListaDinamica) -> None:
    """Serializa la ListaDinamica a un archivo JSON legible por humanos."""
    datos_serializables: list[dict[str, Any]] = []
    actual: Optional[Nodo] = estudiantes._cabeza
    while actual is not None:
        datos_serializables.append(asdict(actual.dato))
        actual = actual.siguiente

    with open(ruta, "w", encoding="utf-8") as archivo:
        json.dump(datos_serializables, archivo, indent=2, ensure_ascii=False)


def cargar_json(ruta: str) -> ListaDinamica:
    """Reconstruye una ListaDinamica de objetos Estudiante desde JSON."""
    with open(ruta, "r", encoding="utf-8") as archivo:
        registros: list[dict[str, Any]] = json.load(archivo)

    lista_recuperada: ListaDinamica = ListaDinamica()
    for reg in registros:
        lista_recuperada.insertar_final(Estudiante(**reg))
    return lista_recuperada


def guardar_pickle(ruta: str, estudiantes: ListaDinamica) -> None:
    """Serializa el grafo completo de objetos (ListaDinamica + Nodos) en binario."""
    with open(ruta, "wb") as archivo:
        pickle.dump(estudiantes, archivo)


def cargar_pickle(ruta: str) -> ListaDinamica:
    """Deserializa directamente la estructura binaria en memoria."""
    with open(ruta, "rb") as archivo:
        return pickle.load(archivo)


# Prueba rápida de ida y vuelta
guardar_json("estudiantes.json", lista_dinamica)
recuperada_json: ListaDinamica = cargar_json("estudiantes.json")

guardar_pickle("estudiantes.pkl", lista_dinamica)
recuperada_pkl: ListaDinamica = cargar_pickle("estudiantes.pkl")

print("Recuperados desde JSON:", len(recuperada_json), "| Primero:", recuperada_json.obtener(0))
print("Recuperados desde Pickle:", len(recuperada_pkl), "| Primero:", recuperada_pkl.obtener(0))